# SRA Regulatory Risk Intelligence  
## 01 — Data Cleaning & Preparation

This notebook prepares a clean, analysis-ready dataset from multiple snapshots of regulatory decisions published by the **Solicitors Regulation Authority (SRA)**.

### Why this step matters
Regulatory datasets collected at different dates often contain repeated records, changing column names, inconsistent category labels, and identifiers stored in unsuitable formats. Before analysing enforcement patterns, those issues need to be handled transparently and reproducibly.

### Cleaning objectives
1. Load all CSV snapshots.
2. Preserve each snapshot's source date and filename.
3. Standardise the decision-date column.
4. Clean text fields and SRA identifiers.
5. Standardise inconsistent decision labels.
6. Identify missing values.
7. Remove repeated regulatory decisions across snapshots.
8. Validate the resulting master dataset.
9. Export a clean CSV for later analysis.

> **Portfolio note:** The raw source data is not embedded in this notebook. The code expects the CSV files in `data/raw/`. This project treats the cleaning methodology and analysis as original work while preserving attribution to the underlying data source.


## 1. Import libraries

In [1]:
from pathlib import Path
import re

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)


## 2. Locate the raw SRA snapshot files

In [2]:
RAW_DATA_DIR = Path("../data/raw")
PROCESSED_DATA_DIR = Path("../data/processed")
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

csv_files = sorted(RAW_DATA_DIR.glob("*_sra_decisions.csv"))

if not csv_files:
    raise FileNotFoundError(
        "No SRA CSV files were found. Place the downloaded snapshot files in data/raw/."
    )

print(f"Files found: {len(csv_files)}")
for file in csv_files:
    print(" -", file.name)


Files found: 9
 - 20260717_sra_decisions.csv
 - 20260807_sra_decisions.csv
 - 20260810_sra_decisions.csv
 - 20260813_sra_decisions.csv
 - 20260817_sra_decisions.csv
 - 20260820_sra_decisions.csv
 - 20260825_sra_decisions.csv
 - 20260827_sra_decisions.csv
 - 20260903_sra_decisions.csv


## 3. Load and combine all snapshots

Each filename begins with the date on which that snapshot was collected. We retain both the filename and the snapshot date so that the origin of every row remains traceable.


In [3]:
frames = []

for file in csv_files:
    df = pd.read_csv(file)

    # The earliest snapshot uses Decision_Date; later snapshots use Date.
    if "Decision_Date" in df.columns:
        df = df.rename(columns={"Decision_Date": "Date"})

    # Capture provenance.
    snapshot_text = file.name[:8]
    df["Snapshot_Date"] = pd.to_datetime(snapshot_text, format="%Y%m%d")
    df["Source_File"] = file.name

    frames.append(df)

raw = pd.concat(frames, ignore_index=True)

print(f"Combined rows: {len(raw):,}")
print(f"Columns: {raw.columns.tolist()}")
raw.head()


Combined rows: 450
Columns: ['Name', 'SRA_ID', 'Decision', 'Date', 'Profile_URL', 'Snapshot_Date', 'Source_File']


,Name,SRA_ID,Decision,Date,Profile_URL,Snapshot_Date,Source_File
0,Robert Bradshaw,117494.0,Intervention,17 July 2026,https://www.sra.org.uk/consumers/solicitor-check/117494,2026-07-17,20260717_sra_decisions.csv
1,Simon Langford,119309.0,Intervention,17 July 2026,https://www.sra.org.uk/consumers/solicitor-check/119309,2026-07-17,20260717_sra_decisions.csv
2,Sayers Solicitors LLP,622585.0,Intervention,17 July 2026,https://www.sra.org.uk/consumers/solicitor-check/622585,2026-07-17,20260717_sra_decisions.csv
3,Hawkins Ryan LLP,8001404.0,Intervention,17 July 2026,https://www.sra.org.uk/consumers/solicitor-check/8001404,2026-07-17,20260717_sra_decisions.csv
4,Yusuf Siddiqui,469225.0,Condition,17 July 2026,https://www.sra.org.uk/consumers/solicitor-check/469225,2026-07-17,20260717_sra_decisions.csv


## 4. Initial data-quality check

In [4]:
quality_before = pd.DataFrame({
    "data_type": raw.dtypes.astype(str),
    "missing_values": raw.isna().sum(),
    "unique_values": raw.nunique(dropna=True)
})

quality_before


,data_type,missing_values,unique_values
Name,object,0,107
SRA_ID,float64,6,106
Decision,object,0,12
Date,object,0,37
Profile_URL,object,0,125
Snapshot_Date,datetime64[ns],0,9
Source_File,object,0,9


### Initial observations

The source snapshots largely share the same schema, but there are several issues worth addressing:

- the decision-date column changes name across snapshots;
- `SRA_ID` contains missing values in some later snapshots;
- numeric parsing can cause SRA IDs to appear with a `.0` suffix;
- the same regulatory decision appears in multiple snapshots;
- at least one decision category has inconsistent capitalisation.

These are common data-engineering issues in periodically collected public datasets.


## 5. Clean identifiers, dates, URLs, and text

In [5]:
clean = raw.copy()

# Parse the regulatory decision date.
clean["Date"] = pd.to_datetime(clean["Date"], errors="coerce")

# Clean whitespace in text fields.
for col in ["Name", "Decision", "Profile_URL"]:
    clean[col] = (
        clean[col]
        .astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

# Treat SRA_ID as an identifier rather than a numeric measure.
# Remove a decimal suffix introduced when CSV parsing encounters missing values.
clean["SRA_ID"] = (
    clean["SRA_ID"]
    .astype("string")
    .str.strip()
    .str.replace(r"\.0$", "", regex=True)
)

clean.head()


,Name,SRA_ID,Decision,Date,Profile_URL,Snapshot_Date,Source_File
0,Robert Bradshaw,117494,Intervention,2026-07-17,https://www.sra.org.uk/consumers/solicitor-check/117494,2026-07-17,20260717_sra_decisions.csv
1,Simon Langford,119309,Intervention,2026-07-17,https://www.sra.org.uk/consumers/solicitor-check/119309,2026-07-17,20260717_sra_decisions.csv
2,Sayers Solicitors LLP,622585,Intervention,2026-07-17,https://www.sra.org.uk/consumers/solicitor-check/622585,2026-07-17,20260717_sra_decisions.csv
3,Hawkins Ryan LLP,8001404,Intervention,2026-07-17,https://www.sra.org.uk/consumers/solicitor-check/8001404,2026-07-17,20260717_sra_decisions.csv
4,Yusuf Siddiqui,469225,Condition,2026-07-17,https://www.sra.org.uk/consumers/solicitor-check/469225,2026-07-17,20260717_sra_decisions.csv


## 6. Standardise decision categories

The source contains the same category with different capitalisation:

- `Regulatory Settlement Agreement`
- `Regulatory settlement agreement`

These should represent one analytical category.


In [6]:
print("Decision labels before standardisation:")
display(clean["Decision"].value_counts().sort_index())

decision_label_map = {
    "Regulatory Settlement Agreement": "Regulatory settlement agreement"
}

clean["Decision"] = clean["Decision"].replace(decision_label_map)

print("\nDecision labels after standardisation:")
display(clean["Decision"].value_counts().sort_index())


Decision labels before standardisation:


Decision
Approval of employment (section 41)                      1
Approval of employment (section 43)                      5
Condition                                               98
Control of non-qualified staff (section 43/99 order)    29
Decisions to refer to SDT                               45
Fine                                                    37
Intervention                                            97
Rebuke                                                  15
Regulatory Settlement Agreement                         54
Regulatory settlement agreement                         41
Solicitors Disciplinary Tribunal order                  25
Termination of suspension of PC/registration             3
Name: count, dtype: Int64


Decision labels after standardisation:


Decision
Approval of employment (section 41)                      1
Approval of employment (section 43)                      5
Condition                                               98
Control of non-qualified staff (section 43/99 order)    29
Decisions to refer to SDT                               45
Fine                                                    37
Intervention                                            97
Rebuke                                                  15
Regulatory settlement agreement                         95
Solicitors Disciplinary Tribunal order                  25
Termination of suspension of PC/registration             3
Name: count, dtype: Int64

## 7. Identify repeated decisions across snapshots

The files are **snapshots**, not nine independent datasets. A decision that remains visible on the SRA source can therefore appear in several later files.

For this project, a regulatory decision is treated as the same event when the following fields match:

- regulated person/entity name;
- decision category;
- decision date; and
- SRA profile URL.

The SRA ID is deliberately **not required** in the deduplication key because some records have a missing `SRA_ID`, while the profile URL still identifies the source record.


In [7]:
dedupe_key = ["Name", "Decision", "Date", "Profile_URL"]

duplicate_mask = clean.duplicated(subset=dedupe_key, keep=False)

print(f"Rows across all snapshots: {len(clean):,}")
print(f"Rows belonging to repeated decisions: {duplicate_mask.sum():,}")
print(f"Unique regulatory decisions: {clean.drop_duplicates(subset=dedupe_key).shape[0]:,}")


Rows across all snapshots: 450
Rows belonging to repeated decisions: 394
Unique regulatory decisions: 126


## 8. Build the master decision dataset

Before removing duplicates, we calculate how many snapshots contained each decision. This preserves useful provenance without counting the same regulatory event multiple times.


In [8]:
snapshot_counts = (
    clean.groupby(dedupe_key, dropna=False)
    .size()
    .rename("Snapshot_Appearances")
    .reset_index()
)

master = (
    clean.sort_values(["Date", "Snapshot_Date"], ascending=[False, False])
    .drop_duplicates(subset=dedupe_key, keep="first")
    .drop(columns=["Snapshot_Date", "Source_File"])
    .merge(snapshot_counts, on=dedupe_key, how="left")
)

# Use a clear, analysis-friendly column order.
master = master[
    ["Name", "SRA_ID", "Decision", "Date", "Profile_URL", "Snapshot_Appearances"]
].sort_values(["Date", "Name"], ascending=[False, True]).reset_index(drop=True)

print(f"Master dataset rows: {len(master):,}")
master.head(10)


Master dataset rows: 126


,Name,SRA_ID,Decision,Date,Profile_URL,Snapshot_Appearances
0,Akbar Ifzal Ali,571258,Decisions to refer to SDT,2026-08-27,https://www.sra.org.uk/consumers/solicitor-check/571258/,2
1,Emma Rowe,279936,Decisions to refer to SDT,2026-08-27,https://www.sra.org.uk/consumers/solicitor-check/279936/,2
2,Heather Joan Roberts,393573,Condition,2026-08-26,https://www.sra.org.uk/consumers/solicitor-check/393573/,2
3,Azair Mahmood Alam,334162,Intervention,2026-08-25,https://www.sra.org.uk/consumers/solicitor-check/334162/,2
4,Lewis Duncan - 3,782329,Control of non-qualified staff (section 43/99 order),2026-08-25,https://www.sra.org.uk/consumers/solicitor-check/7823293/,2
5,Roundhay Solicitors,649410,Intervention,2026-08-25,https://www.sra.org.uk/consumers/solicitor-check/649410/,2
6,Siobhan Hall (Cook) - 3,781770,Control of non-qualified staff (section 43/99 order),2026-08-24,https://www.sra.org.uk/consumers/solicitor-check/7817703/,3
7,Jeremy Brooke,202554,Decisions to refer to SDT,2026-08-20,https://www.sra.org.uk/consumers/solicitor-check/202554/,4
8,Michael G Wooldridge,641191,Intervention,2026-08-19,https://www.sra.org.uk/consumers/solicitor-check/641191/,4
9,Michael George Wooldridge,106907,Intervention,2026-08-19,https://www.sra.org.uk/consumers/solicitor-check/106907/,4


## 9. Validate the cleaned dataset

In [9]:
validation = pd.Series({
    "master_rows": len(master),
    "remaining_duplicate_events": master.duplicated(subset=dedupe_key).sum(),
    "missing_names": master["Name"].isna().sum(),
    "missing_decisions": master["Decision"].isna().sum(),
    "missing_dates": master["Date"].isna().sum(),
    "missing_profile_urls": master["Profile_URL"].isna().sum(),
    "missing_sra_ids": master["SRA_ID"].isna().sum(),
    "decision_categories": master["Decision"].nunique()
}, name="value")

validation.to_frame()


,value
master_rows,126
remaining_duplicate_events,0
missing_names,0
missing_decisions,0
missing_dates,0
missing_profile_urls,0
missing_sra_ids,1
decision_categories,11


## 10. Review the cleaned decision categories

In [10]:
decision_summary = (
    master["Decision"]
    .value_counts()
    .rename_axis("Decision")
    .reset_index(name="Count")
)

decision_summary["Share_Percent"] = (
    decision_summary["Count"] / len(master) * 100
).round(1)

decision_summary


,Decision,Count,Share_Percent
0,Condition,31,24.6
1,Regulatory settlement agreement,21,16.7
2,Intervention,20,15.9
3,Decisions to refer to SDT,16,12.7
4,Control of non-qualified staff (section 43/99 order),12,9.5
5,Fine,11,8.7
6,Solicitors Disciplinary Tribunal order,6,4.8
7,Rebuke,4,3.2
8,Termination of suspension of PC/registration,3,2.4
9,Approval of employment (section 43),1,0.8


## 11. Export the analysis-ready dataset

In [11]:
output_path = PROCESSED_DATA_DIR / "sra_decisions_clean.csv"
master.to_csv(output_path, index=False)

print(f"Clean dataset saved to: {output_path}")
print(f"Rows exported: {len(master):,}")


Clean dataset saved to: ../data/processed/sra_decisions_clean.csv
Rows exported: 126


## Cleaning summary

The preparation pipeline converts repeated SRA snapshot files into one reproducible master table of unique regulatory decisions.

The cleaned dataset:

- standardises the source schema;
- preserves source provenance during processing;
- parses decision dates consistently;
- treats SRA IDs as identifiers rather than numerical variables;
- normalises inconsistent decision labels;
- removes repeated decisions created by overlapping snapshots;
- records the number of snapshots in which each decision appeared; and
- produces a validated CSV for exploratory analysis.

### Next notebook
`02_exploratory_analysis.ipynb` will use the cleaned dataset to examine enforcement patterns, decision categories, timing, repeat appearances, and other features relevant to regulatory-risk analysis.

### Methodological limitation
This dataset reflects the decisions captured in the downloaded snapshots. It should not be interpreted as a complete census of all SRA enforcement activity or as evidence that a listed person or firm presents current regulatory risk. Later risk indicators in this portfolio will be explicitly labelled as **analytical prototypes**, not official SRA assessments.
